# apmc_page_split.ipynb -- two-pass vision-based catalog PDF splitter

Splits an ASCC catalog PDF into marking-anchored half-page chunks using
two passes of Sonnet vision calls.

## Inputs / outputs

- input PDF:        `wip/in/{BASENAME}.pdf`
- rendered pages:   `wip/cache/{BASENAME}_full/page-NNNN.png`     (NNNN = PDF index)
- pass-1 halves:    `wip/cache/{BASENAME}_halves/page-NNNN-{L,R}.png` (NNNN = catalog page)
- pass-1 cache:     `wip/cache/{BASENAME}_split_halves.json`
- pass-2 cache:     `wip/cache/{BASENAME}_split_chunks.json`
- final chunks:     `wip/out/{BASENAME}/page-NNNN-{L,R}-K.png`     (or `page-NNNN-K.png` for single-col)
- manifest CSV:     `wip/out/{BASENAME}_split_manifest.csv`

## Pipeline

1. Render PDF with pdftoppm at 300 DPI to `FULL_DIR` (cached).
2. Pass 1 (one Sonnet call per full page): detect column gutter and printed catalog page number.
3. Crop each full page into L/R halves using detected gutter; write to `HALVES_DIR` named by catalog page.
4. Pass 2 (one Sonnet call per half image): detect y-cuts in the half's local coords, anchored to postal markings.
5. Crop each half into chunks using filtered cuts; write to `OUTPUT_DIR`.
6. Emit manifest CSV.

## Downstream

`apmc_page_extract.ipynb` currently expects `page-NNNN-{L,R}.png`. New filenames are
`page-NNNN-{L,R}-K.png` and `page-NNNN-K.png`. Update its NAME_RE and per-page iteration
before running it against this output.

In [1]:
import base64
import json
import os
import re
import shutil
import subprocess
import sys
from io import BytesIO
from pathlib import Path

import pandas as pd
from PIL import Image
from dotenv import load_dotenv
import anthropic

load_dotenv(Path('..') / '.env')

BASENAME             = 'VA_ASCC_CTLG'
PDF_PATH             = Path(f'./wip/in/{BASENAME}.pdf')
FULL_DIR             = Path(f'./wip/cache/{BASENAME}_full')
HALVES_DIR           = Path(f'./wip/cache/{BASENAME}_halves')
HALVES_CACHE         = Path(f'./wip/cache/{BASENAME}_split_halves.json')
CHUNKS_CACHE         = Path(f'./wip/cache/{BASENAME}_split_chunks.json')
OUTPUT_DIR           = Path(f'./wip/out/{BASENAME}')
MANIFEST_CSV         = Path(f'./wip/out/{BASENAME}_split_manifest.csv')

DPI                  = 300
GUTTER_PAD_PX        = 12
MIN_CHUNK_HEIGHT_PX  = 200

# Pass 1 strip sizes (300 DPI -> 1 inch = 300 px).
GUTTER_STRIP_WIDTH   = 400   # ~1.3" wide vertical strip centered at width//2
HEADER_STRIP_HEIGHT  = 250   # ~0.83" tall band from top
FOOTER_STRIP_HEIGHT  = 250   # ~0.83" tall band from bottom
HEADER_FOOTER_SEP_PX = 6     # black separator between header and footer in composite

# Pass 1: page-number call + gutter-strip call. Sonnet handles both -- each task
# is small and constrained. Cache invalidates on prompt-version bump.
HALVES_MODEL         = 'claude-sonnet-4-6'
HALVES_PROMPT_VER    = 'v2'  # v2 = strip-based pass-1 design
HALVES_MAX_TOKENS    = 128

CHUNKS_MODEL         = 'claude-opus-4-7'
CHUNKS_PROMPT_VER    = 'v1'
CHUNKS_MAX_TOKENS    = 256

FORCE_REFRESH_RENDER = False
FORCE_REFRESH_HALVES = False
FORCE_REFRESH_CHUNKS = False

assert PDF_PATH.is_file(), f'missing {PDF_PATH}'
assert shutil.which('pdftoppm'), 'pdftoppm not on PATH (install poppler)'
assert os.environ.get('ANTHROPIC_API_KEY'), 'ANTHROPIC_API_KEY not set'

for d in (FULL_DIR, HALVES_DIR, OUTPUT_DIR,
          HALVES_CACHE.parent, CHUNKS_CACHE.parent):
    d.mkdir(parents=True, exist_ok=True)

client = anthropic.Anthropic()
print(f'PDF:           {PDF_PATH}')
print(f'FULL_DIR:      {FULL_DIR}')
print(f'HALVES:        {HALVES_DIR}')
print(f'OUT:           {OUTPUT_DIR}')
print(f'HALVES_MODEL:  {HALVES_MODEL}  ({HALVES_PROMPT_VER})')
print(f'CHUNKS_MODEL:  {CHUNKS_MODEL}  ({CHUNKS_PROMPT_VER})')


PDF:           wip/in/VA_ASCC_CTLG.pdf
FULL_DIR:      wip/cache/VA_ASCC_CTLG_full
HALVES:        wip/cache/VA_ASCC_CTLG_halves
OUT:           wip/out/VA_ASCC_CTLG
HALVES_MODEL:  claude-sonnet-4-6  (v2)
CHUNKS_MODEL:  claude-opus-4-7  (v1)


In [2]:
# Render PDF -> wip/cache/{BASENAME}_full/page-NNNN.png
# NNNN is the PDF order index, zero-padded to 4 digits.
# Skip if FULL_DIR already has page-*.png and not FORCE_REFRESH_RENDER.

def _idx(p):
    m = re.search(r'-(\d+)\.png$', p.name)
    return int(m.group(1)) if m else 0

def render_pdf(pdf_path, full_dir, dpi):
    # pdftoppm writes <prefix>-N.png un-padded for low N -- render to a sentinel
    # prefix, then rename to zero-padded page-NNNN.png in numeric order.
    for old in full_dir.glob('_render-*.png'):
        old.unlink()
    prefix = full_dir / '_render'
    subprocess.run(
        ['pdftoppm', '-r', str(dpi), '-png', str(pdf_path), str(prefix)],
        check=True,
    )
    rendered = sorted(full_dir.glob('_render-*.png'), key=_idx)
    pages = []
    for i, src in enumerate(rendered, 1):
        dst = full_dir / f'page-{i:04d}.png'
        if dst.exists():
            dst.unlink()
        src.rename(dst)
        pages.append(dst)
    return pages

existing = sorted(FULL_DIR.glob('page-*.png'), key=_idx)
if existing and not FORCE_REFRESH_RENDER:
    print(f'render: {len(existing)} pages already in {FULL_DIR}, skipping pdftoppm')
    full_pages = existing
else:
    print(f'render: pdftoppm -r {DPI} {PDF_PATH.name} -> {FULL_DIR}')
    full_pages = render_pdf(PDF_PATH, FULL_DIR, DPI)
    print(f'render: wrote {len(full_pages)} pages')

assert full_pages, 'no rendered pages -- check PDF and pdftoppm output'


render: pdftoppm -r 300 VA_ASCC_CTLG.pdf -> wip/cache/VA_ASCC_CTLG_full
render: wrote 17 pages


## Pass 1 -- page number + gutter (two sub-calls per page)

Two narrow sub-tasks instead of one wide one:

1. **Page number** -- send a composite of the top `HEADER_STRIP_HEIGHT` px and bottom `FOOTER_STRIP_HEIGHT` px
   of the page (stacked, separated by a thin black bar). Ask only for `{page_number: int}`.
2. **Gutter** -- pre-crop a `GUTTER_STRIP_WIDTH`-wide vertical strip centered on `width // 2`. Send only that
   strip. Ask for `{has_two_columns, left_edge, right_edge}` in the strip's LOCAL x-coords. We add the strip's
   x-offset back to map into full-page coords before caching.

Anchoring the gutter task at the midpoint reduces it to a small bounded delta search, which Sonnet handles.
Frozen system prompts sent with `cache_control: ephemeral`.

In [3]:
PAGE_NUMBER_SYSTEM_PROMPT = '''You receive a stitched image showing the TOP margin and BOTTOM margin of one page of an old American Stampless Cover (ASCC) catalog. The two margins are stacked vertically with a black separator bar between them.

Your job is to read the printed catalog page number. It appears as a plain integer somewhere in either margin -- usually in the footer, occasionally in the header.

Return STRICT JSON only -- no markdown, no prose, no code fences:

  {"page_number": 419}

Rules:
- page_number is an integer >= 1.
- Never return null. If you cannot read it confidently, return your best integer guess.
- Output JSON only.'''

GUTTER_SYSTEM_PROMPT = '''You receive a NARROW VERTICAL STRIP cropped from the MIDDLE of one page of an old American Stampless Cover (ASCC) catalog. The strip is approximately STRIP_WIDTH pixels wide and full page height tall, cut centered on the page horizontal midpoint.

For most pages this strip contains, from left to right:
- the rightmost glyphs of the LEFT text column (along the strip left side)
- a vertical white GUTTER between the columns
- the leftmost glyphs of the RIGHT text column (along the strip right side)

Your job is to locate the gutter and return its boundaries as x-pixels in THIS STRIP local coordinate system (origin top-left of the strip, NOT the full page).

Return STRICT JSON only -- no markdown, no prose, no code fences:

  {"has_two_columns": true, "left_edge": 145, "right_edge": 195}

Or, if the strip shows continuous text, solid whitespace, or no clear vertical gutter (single-column page, full-page plate, blank page, section divider):

  {"has_two_columns": false, "left_edge": -1, "right_edge": -1}

Rules:
- left_edge: x-pixel where the LEFT column ENDS. A few pixels past the rightmost glyph of the left column, inside the white gutter.
- right_edge: x-pixel where the RIGHT column BEGINS. A few pixels before the leftmost glyph of the right column, inside the white gutter.
- Both are integers in the strip local x-coords. When has_two_columns is true: 0 <= left_edge < right_edge <= strip_width.
- When has_two_columns is false, set both edges to -1.
- Output JSON only.'''

print(f'page-number prompt: {len(PAGE_NUMBER_SYSTEM_PROMPT)} chars')
print(f'gutter prompt:      {len(GUTTER_SYSTEM_PROMPT)} chars')


page-number prompt: 618 chars
gutter prompt:      1490 chars


In [4]:
def _img_to_b64_png(im):
    buf = BytesIO()
    im.save(buf, format='PNG')
    return base64.standard_b64encode(buf.getvalue()).decode()

def build_header_footer_strip(im):
    # Composite of top HEADER_STRIP_HEIGHT and bottom FOOTER_STRIP_HEIGHT pixels,
    # stacked vertically, separated by a HEADER_FOOTER_SEP_PX-tall black bar.
    w, h = im.size
    head = im.crop((0, 0, w, min(h, HEADER_STRIP_HEIGHT)))
    foot = im.crop((0, max(0, h - FOOTER_STRIP_HEIGHT), w, h))
    out_h = head.size[1] + HEADER_FOOTER_SEP_PX + foot.size[1]
    out = Image.new('RGB', (w, out_h), (0, 0, 0))
    out.paste(head, (0, 0))
    out.paste(foot, (0, head.size[1] + HEADER_FOOTER_SEP_PX))
    return out

def build_gutter_strip(im):
    # Vertical strip GUTTER_STRIP_WIDTH wide centered on width//2, full height.
    # Returns (strip_image, x_offset_in_full_page) so caller can map strip-local
    # coords back to full-page coords by adding x_offset.
    w, h = im.size
    cx = w // 2
    half = GUTTER_STRIP_WIDTH // 2
    x0 = max(0, cx - half)
    x1 = min(w, cx + half)
    strip = im.crop((x0, 0, x1, h))
    return strip, x0

def _parse_strict_json(resp):
    text = resp.content[0].text.strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text, flags=re.DOTALL)
    return json.loads(text)

def detect_page_number(header_footer_im):
    img_b64 = _img_to_b64_png(header_footer_im)
    user = [
        {'type': 'image',
         'source': {'type': 'base64', 'media_type': 'image/png', 'data': img_b64}},
        {'type': 'text', 'text': 'Read the printed catalog page number. Return JSON only.'},
    ]
    resp = client.messages.create(
        model=HALVES_MODEL,
        max_tokens=HALVES_MAX_TOKENS,
        system=[{'type': 'text', 'text': PAGE_NUMBER_SYSTEM_PROMPT,
                 'cache_control': {'type': 'ephemeral'}}],
        messages=[{'role': 'user', 'content': user}],
    )
    data = _parse_strict_json(resp)
    pn = data['page_number']
    assert isinstance(pn, int) and pn >= 1, f'bad page_number {pn!r}'
    return pn

def detect_gutter(strip_im):
    strip_w = strip_im.size[0]
    img_b64 = _img_to_b64_png(strip_im)
    user = [
        {'type': 'image',
         'source': {'type': 'base64', 'media_type': 'image/png', 'data': img_b64}},
        {'type': 'text',
         'text': f'Strip width: {strip_w} px. Return JSON only.'},
    ]
    resp = client.messages.create(
        model=HALVES_MODEL,
        max_tokens=HALVES_MAX_TOKENS,
        system=[{'type': 'text', 'text': GUTTER_SYSTEM_PROMPT,
                 'cache_control': {'type': 'ephemeral'}}],
        messages=[{'role': 'user', 'content': user}],
    )
    data = _parse_strict_json(resp)
    htc = data['has_two_columns']
    le = data['left_edge']
    re_ = data['right_edge']
    assert isinstance(htc, bool), f'bad has_two_columns {htc!r}'
    assert isinstance(le, int) and isinstance(re_, int), f'bad edges {le!r},{re_!r}'
    if htc:
        assert 0 <= le < re_ <= strip_w, (
            f'bad two-col edges {le},{re_} vs strip_width {strip_w}'
        )
    else:
        if le != -1 or re_ != -1:
            print(f'  WARNING: has_two_columns=false but edges={le},{re_}; coercing to -1,-1')
            le, re_ = -1, -1
    return htc, le, re_


In [5]:
# Pass 1 cache schema (left_edge / right_edge are FULL-PAGE x-coords, mapped from strip-local):
#   {
#     "model": "claude-sonnet-4-6",
#     "prompt_version": "v2",
#     "responses": {
#       "pdf-page-0001": {"page_number": 419, "has_two_columns": true,
#                         "left_edge": 1240, "right_edge": 1290,
#                         "image_width": 2550, "image_height": 3300,
#                         "strip_x_offset": 1075, "strip_width": 400,
#                         "left_edge_strip": 165, "right_edge_strip": 215}
#     }
#   }

if HALVES_CACHE.exists() and not FORCE_REFRESH_HALVES:
    halves_cache = json.loads(HALVES_CACHE.read_text())
    if (halves_cache.get('model') != HALVES_MODEL
            or halves_cache.get('prompt_version') != HALVES_PROMPT_VER):
        print('halves cache: model/prompt_version mismatch, invalidating')
        halves_cache = {'model': HALVES_MODEL, 'prompt_version': HALVES_PROMPT_VER, 'responses': {}}
else:
    halves_cache = {'model': HALVES_MODEL, 'prompt_version': HALVES_PROMPT_VER, 'responses': {}}

calls = 0
for pdf_idx, full_png in enumerate(full_pages, 1):
    key = f'pdf-page-{pdf_idx:04d}'
    with Image.open(full_png) as im:
        iw, ih = im.size
        if key in halves_cache['responses'] and not FORCE_REFRESH_HALVES:
            rec = halves_cache['responses'][key]
        else:
            hf_strip   = build_header_footer_strip(im)
            gut_strip, x_off = build_gutter_strip(im)
            sw = gut_strip.size[0]

            pn = detect_page_number(hf_strip)
            calls += 1
            htc, le_strip, re_strip = detect_gutter(gut_strip)
            calls += 1

            if htc:
                le_full = x_off + le_strip
                re_full = x_off + re_strip
                assert 0 < le_full < re_full < iw, (
                    f'mapped edges {le_full},{re_full} out of (0, {iw})'
                )
            else:
                le_full = -1
                re_full = -1

            rec = {
                'page_number':       pn,
                'has_two_columns':   htc,
                'left_edge':         le_full,
                'right_edge':        re_full,
                'image_width':       iw,
                'image_height':      ih,
                'strip_x_offset':    x_off,
                'strip_width':       sw,
                'left_edge_strip':   le_strip,
                'right_edge_strip':  re_strip,
            }
            halves_cache['responses'][key] = rec
            HALVES_CACHE.write_text(json.dumps(halves_cache, indent=2))

    pn = rec['page_number']
    if rec['has_two_columns']:
        print(f'  {key} -> catalog {pn:>4d}  edges_full=[{rec["left_edge"]},{rec["right_edge"]}]  '
              f'(strip=[{rec["left_edge_strip"]},{rec["right_edge_strip"]}] @ x_off={rec["strip_x_offset"]})  '
              f'size {iw}x{ih}')
    else:
        print(f'  {key} -> catalog {pn:>4d}  SINGLE-COLUMN  size {iw}x{ih}')

print(f'pass-1 calls made: {calls}')


  pdf-page-0001 -> catalog  419  edges_full=[987,1022]  (strip=[175,210] @ x_off=812)  size 2024x2970
  pdf-page-0002 -> catalog  420  edges_full=[967,987]  (strip=[155,175] @ x_off=812)  size 2024x2970
  pdf-page-0003 -> catalog  421  edges_full=[1002,1028]  (strip=[192,218] @ x_off=810)  size 2020x2970
  pdf-page-0004 -> catalog  422  edges_full=[928,952]  (strip=[118,142] @ x_off=810)  size 2020x2970
  pdf-page-0005 -> catalog  423  edges_full=[1007,1027]  (strip=[195,215] @ x_off=812)  size 2025x2965
  pdf-page-0006 -> catalog  424  edges_full=[967,987]  (strip=[155,175] @ x_off=812)  size 2025x2965
  pdf-page-0007 -> catalog  425  edges_full=[1007,1032]  (strip=[195,220] @ x_off=812)  size 2025x2968
  pdf-page-0008 -> catalog  426  edges_full=[1006,1030]  (strip=[194,218] @ x_off=812)  size 2025x2968
  pdf-page-0009 -> catalog  427  edges_full=[1007,1031]  (strip=[194,218] @ x_off=813)  size 2026x2966
  pdf-page-0010 -> catalog  428  edges_full=[923,968]  (strip=[110,155] @ x_off=

In [6]:
# Crop full pages into L/R halves (or full single-col pages) using cached gutter coords.
# Wipes HALVES_DIR/page-*.png first so stale halves do not linger across re-runs.
# Detects duplicate catalog page numbers BEFORE any writes -- raises with offenders listed.

by_pn = {}
for pdf_idx, full_png in enumerate(full_pages, 1):
    key = f'pdf-page-{pdf_idx:04d}'
    pn = halves_cache['responses'][key]['page_number']
    by_pn.setdefault(pn, []).append(key)
dups = {pn: keys for pn, keys in by_pn.items() if len(keys) > 1}
if dups:
    raise RuntimeError(
        f'duplicate catalog page_number(s) detected in pass-1 cache: {dups}.\n'
        f'Edit {HALVES_CACHE} to fix the misread page_number(s) and re-run this cell.'
    )

for old in HALVES_DIR.glob('page-*.png'):
    old.unlink()

halves_written = 0
for pdf_idx, full_png in enumerate(full_pages, 1):
    key = f'pdf-page-{pdf_idx:04d}'
    rec = halves_cache['responses'][key]
    pn = rec['page_number']
    with Image.open(full_png) as im:
        w, h = im.size
        if rec['has_two_columns']:
            l_x1 = min(w, rec['left_edge']  + GUTTER_PAD_PX)
            r_x0 = max(0, rec['right_edge'] - GUTTER_PAD_PX)
            im.crop((0,    0, l_x1, h)).save(HALVES_DIR / f'page-{pn:04d}-L.png')
            im.crop((r_x0, 0, w,    h)).save(HALVES_DIR / f'page-{pn:04d}-R.png')
            halves_written += 2
        else:
            im.crop((0, 0, w, h)).save(HALVES_DIR / f'page-{pn:04d}.png')
            halves_written += 1

print(f'wrote {halves_written} half images to {HALVES_DIR}')
print('--- INSPECT THESE BEFORE PROCEEDING TO PASS 2 ---')
print('Open a few halves and confirm no listing text or marking is clipped at the seam.')


wrote 34 half images to wip/cache/VA_ASCC_CTLG_halves
--- INSPECT THESE BEFORE PROCEEDING TO PASS 2 ---
Open a few halves and confirm no listing text or marking is clipped at the seam.


## Pass 2 -- chunk cuts per half

One Sonnet vision call per half image. Returns `{cuts: [int, ...]}` in that half's local y-coord system.
Markings sit at the TOP of the chunk below each cut. Empty list when no cuts produce non-trivial chunks.

In [7]:
CHUNKS_SYSTEM_PROMPT = '''You are a layout analyzer for ONE column-image from an old American Stampless Cover (ASCC) catalog.

This is a STAMPLESS COVER catalog. The images on these pages are postal markings -- postmarks,
handstamps, and manuscript markings. They are NEVER adhesive postage stamps.

You will receive ONE half-column image (or a full single-column page). Your job is to find clean
horizontal cut lines so that downstream extraction can process the column as smaller chunks, with
each postal marking sitting at the TOP of the chunk below its cut.

Return STRICT JSON only -- no markdown, no prose, no code fences -- of this exact shape:

  {"cuts": [820, 1640]}

Or, for an image with no valid cuts:

  {"cuts": []}

Cut placement rules:

- Examine the image top-to-bottom. For each distinct postal-marking reproduction, find the y-pixel
  of a clean horizontal line in the WHITE SPACE just above the marking top edge -- between the
  bottom of the prior listing text and the marking top.

- Counting rule: one marking reproduction is ONE image. Small adjacent date or rate annotations
  printed next to the marking are part of the SAME image, not separate ones. Do not place a cut
  between a marking and its inline annotations.

- Coordinate system: y-pixels are integers in the LOCAL coordinate system of THIS image, with
  origin top-left. All cuts must be strictly inside (0, image_height) and strictly ascending.

Non-trivial-chunk rule (CRITICAL -- read carefully):

  Only emit a cut if it produces two NON-TRIVIAL chunks. A chunk is non-trivial when it contains
  REAL LISTING TEXT, not just whitespace or an isolated marking.

  Concretely: emit a cut above a marking only when there is at least one full listing of text
  ABOVE the cut AND at least the marking plus one full listing of text BELOW the cut.

  Examples:

  - Marking flush against the TOP of the image (no listings above it on this image):
    emit NO cut for that marking. It already sits at the top of chunk 1 naturally.

  - Marking flush against the BOTTOM of the image (no listings below it on this image):
    emit NO cut for that marking. It already sits inside the trailing chunk tail.

  - Two markings back-to-back with only whitespace between them:
    emit ONE cut above the FIRST marking. Do NOT emit a second cut between them, since that
    would create a chunk containing only the first marking and no listing text.

  If you would create a chunk with no listing text, do not emit the cut.

Output JSON only.'''

print(f'pass-2 prompt: {len(CHUNKS_SYSTEM_PROMPT)} chars')


pass-2 prompt: 2489 chars


In [8]:
def detect_cuts(image_path, image_height):
    img_b64 = base64.standard_b64encode(image_path.read_bytes()).decode()
    user = [
        {'type': 'image',
         'source': {'type': 'base64', 'media_type': 'image/png', 'data': img_b64}},
        {'type': 'text',
         'text': f'Image height: {image_height} px. Return JSON only.'},
    ]
    resp = client.messages.create(
        model=CHUNKS_MODEL,
        max_tokens=CHUNKS_MAX_TOKENS,
        system=[{'type': 'text', 'text': CHUNKS_SYSTEM_PROMPT,
                 'cache_control': {'type': 'ephemeral'}}],
        messages=[{'role': 'user', 'content': user}],
    )
    text = resp.content[0].text.strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text, flags=re.DOTALL)
    data = json.loads(text)
    cuts = data['cuts']
    assert isinstance(cuts, list), f'cuts not a list: {cuts!r}'
    for i, c in enumerate(cuts):
        assert isinstance(c, int), f'cuts[{i}] not int: {c!r}'
        assert 0 < c < image_height, f'cuts[{i}]={c} out of (0, {image_height})'
    assert cuts == sorted(cuts), f'cuts not strictly ascending: {cuts}'
    for a, b in zip(cuts, cuts[1:]):
        assert a != b, f'cuts has duplicate: {cuts}'
    return cuts


In [9]:
# Pass 2 cache schema:
#   {
#     "model": "claude-opus-4-7",
#     "prompt_version": "v1",
#     "responses": {
#       "page-0419-L": {"cuts": [820, 1640], "image_height": 3300},
#       "page-0419-R": {"cuts": [350],       "image_height": 3300},
#       "page-0250":   {"cuts": [],          "image_height": 3300}
#     }
#   }

if CHUNKS_CACHE.exists() and not FORCE_REFRESH_CHUNKS:
    chunks_cache = json.loads(CHUNKS_CACHE.read_text())
    if (chunks_cache.get('model') != CHUNKS_MODEL
            or chunks_cache.get('prompt_version') != CHUNKS_PROMPT_VER):
        print('chunks cache: model/prompt_version mismatch, invalidating')
        chunks_cache = {'model': CHUNKS_MODEL, 'prompt_version': CHUNKS_PROMPT_VER, 'responses': {}}
else:
    chunks_cache = {'model': CHUNKS_MODEL, 'prompt_version': CHUNKS_PROMPT_VER, 'responses': {}}

halves = sorted(HALVES_DIR.glob('page-*.png'))
assert halves, f'no halves under {HALVES_DIR} -- run pass-1 cells first'

calls = 0
for half_png in halves:
    stem = half_png.stem
    with Image.open(half_png) as im:
        ih = im.size[1]
    if stem in chunks_cache['responses'] and not FORCE_REFRESH_CHUNKS:
        rec = chunks_cache['responses'][stem]
    else:
        cuts = detect_cuts(half_png, ih)
        rec = {'cuts': cuts, 'image_height': ih}
        chunks_cache['responses'][stem] = rec
        CHUNKS_CACHE.write_text(json.dumps(chunks_cache, indent=2))
        calls += 1
    print(f'  {stem}  cuts={rec["cuts"]}')

print(f'pass-2 calls made: {calls}')


  page-0419-L  cuts=[700, 1030, 1340, 1610, 1830, 2120, 2300, 2500]
  page-0419-R  cuts=[300, 540, 860, 1140, 1500, 1810, 2150, 2570]
  page-0420-L  cuts=[480, 810, 1230, 1730, 2080, 2310, 2570]
  page-0420-R  cuts=[330, 560, 790, 1100, 1300, 1530, 1810, 2030, 2240, 2470, 2680]
  page-0421-L  cuts=[855, 1645, 2095, 2475]
  page-0421-R  cuts=[770, 1450, 1900, 2480]
  page-0422-L  cuts=[230, 580, 1090, 1670, 2090, 2400]
  page-0422-R  cuts=[460, 860, 1450, 1850, 2250]
  page-0423-L  cuts=[400, 890, 1230, 1610, 2090, 2390, 2670]
  page-0423-R  cuts=[210, 805, 1290, 1810, 2480]


KeyboardInterrupt: 

In [ ]:
# Crop each half into chunks using filtered cuts.
# Wipes OUTPUT_DIR/*.png first so stale chunks do not linger across re-runs.

def filter_cuts(cuts, image_height, label):
    kept = []
    last = 0
    for c in cuts:
        gap_above = c - last
        gap_below = image_height - c
        if gap_above < MIN_CHUNK_HEIGHT_PX:
            print(f'    [{label}] drop cut y={c}: gap above={gap_above} < {MIN_CHUNK_HEIGHT_PX}')
            continue
        if gap_below < MIN_CHUNK_HEIGHT_PX:
            print(f'    [{label}] drop cut y={c}: gap below={gap_below} < {MIN_CHUNK_HEIGHT_PX}')
            continue
        kept.append(c)
        last = c
    return kept

def write_chunks(half_png, raw_cuts, out_dir):
    stem = half_png.stem
    with Image.open(half_png) as im:
        w, h = im.size
        kept = filter_cuts(raw_cuts, h, stem)
        ys = [0] + kept + [h]
        for k, (y0, y1) in enumerate(zip(ys[:-1], ys[1:]), 1):
            im.crop((0, y0, w, y1)).save(out_dir / f'{stem}-{k}.png')
    return kept, len(ys) - 1

for old in OUTPUT_DIR.glob('*.png'):
    old.unlink()

summary = []
total_chunks = 0
for half_png in halves:
    stem = half_png.stem
    rec = chunks_cache['responses'][stem]
    kept, n = write_chunks(half_png, rec['cuts'], OUTPUT_DIR)
    total_chunks += n
    summary.append({'stem': stem, 'raw_cuts': rec['cuts'], 'kept_cuts': kept,
                    'num_chunks': n, 'image_height': rec['image_height']})

print(f'wrote {total_chunks} chunk PNGs across {len(halves)} halves -> {OUTPUT_DIR}')


In [ ]:
# Manifest CSV: one row per half image (or per page for single-col).
#   pdf_page_index, catalog_page, side, has_two_columns, left_edge, right_edge,
#   half_width, half_height, raw_cuts, kept_cuts, num_chunks

STEM_RE = re.compile(r'^page-(\d{4})(?:-([LR]))?$')

pn_to_pdf_idx = {}
for pdf_idx, full_png in enumerate(full_pages, 1):
    key = f'pdf-page-{pdf_idx:04d}'
    pn_to_pdf_idx[halves_cache['responses'][key]['page_number']] = pdf_idx

rows = []
for s in summary:
    m = STEM_RE.match(s['stem'])
    assert m, f'unexpected stem {s["stem"]}'
    pn = int(m.group(1))
    side = m.group(2) or ''
    pdf_idx = pn_to_pdf_idx[pn]
    h_rec = halves_cache['responses'][f'pdf-page-{pdf_idx:04d}']
    with Image.open(HALVES_DIR / f'{s["stem"]}.png') as im:
        hw, hh = im.size
    rows.append({
        'pdf_page_index':  pdf_idx,
        'catalog_page':    pn,
        'side':            side,
        'has_two_columns': h_rec['has_two_columns'],
        'left_edge':       h_rec['left_edge'],
        'right_edge':      h_rec['right_edge'],
        'half_width':      hw,
        'half_height':     hh,
        'raw_cuts':        ';'.join(str(c) for c in s['raw_cuts']),
        'kept_cuts':       ';'.join(str(c) for c in s['kept_cuts']),
        'num_chunks':      s['num_chunks'],
    })

df = pd.DataFrame(rows).sort_values(['pdf_page_index', 'side']).reset_index(drop=True)
df.to_csv(MANIFEST_CSV, index=False)
print(f'manifest: {MANIFEST_CSV}  ({len(df)} rows)')
df.head(10)


## Verification

- **Pass-1 sanity** (after cell a5): `pass-1 calls made: 0` on re-runs. Open a few entries in
  `wip/cache/{BASENAME}_halves/` and confirm no listing text or postal marking is clipped at
  the L/R seam BEFORE letting pass 2 spend tokens.
- **Pass-2 sanity** (after cell a9): `pass-2 calls made: 0` on re-runs. Spot-check chunks under
  `wip/out/{BASENAME}/`:
  - For each `page-NNNN-{L,R}-K` with K >= 2, the corresponding postmark/handstamp/manuscript
    marking should sit at the TOP of the image, with listings flowing below it.
  - Chunk K=1 should END with listing text, not contain a marking already.
  - No chunk should be mostly whitespace (empty-chunk guard working).
  - Per-half chunk counts should be sane (typically 1-5, not 10).
- **Manifest sanity**: `catalog_page` strictly monotonic across `pdf_page_index`. Any non-monotonic
  row = pass-1 page-number misread; edit the entry in `_split_halves.json` by hand and re-run from
  cell a6 onward (cropping reads from cache, no API calls).
- **Re-tuning without API spend**: change `MIN_CHUNK_HEIGHT_PX` or `GUTTER_PAD_PX`, then re-run
  cell a6 (halves) and/or cell a10 (chunks). No vision calls happen.
- **Downstream refactor (out of scope)**: `apmc_page_extract.ipynb` currently uses
  `NAME_RE = r"^page-(\d{4})-([LR])\.png$"`. New filenames are `page-(\d{4})-([LR])-(\d+).png`
  (two-column) and `page-(\d{4})-(\d+).png` (single-column). Update the regex and per-page
  iteration to walk `(catalog_page, side, chunk_k)` order before running it against this output.